In [ ]:
import os

# Gensim provides Word2Vec and KeyedVectors utilities for pretrained vectors
import gensim
from gensim.models import Word2Vec, KeyedVectors
import gensim.downloader as api

# Note: The notebook's working directory is the `notebooks` folder.
# We'll save large pretrained files to the sibling `../models` folder to
# keep them separate from source and to avoid accidentally committing them to Git.


In [ ]:
# Download the model only if it's not already present locally.
# The pretrained `word2vec-google-news-300` model is ~1.5–1.6GB.
MODEL_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', 'models'))
os.makedirs(MODEL_DIR, exist_ok=True)

# Two paths: a gensim-native KeyedVectors file (fast to load) and
# the original binary format (for compatibility with other tools).
KV_PATH = os.path.join(MODEL_DIR, 'word2vec_google_news_300.kv')
BIN_PATH = os.path.join(MODEL_DIR, 'GoogleNews-vectors-negative300.bin')

# Use existing saved files when possible to avoid re-downloading.
if os.path.exists(KV_PATH):
    print('Found saved KeyedVectors at', KV_PATH)
    wv = KeyedVectors.load(KV_PATH, mmap='r')

elif os.path.exists(BIN_PATH):
    print('Found binary word2vec file at', BIN_PATH)
    wv = KeyedVectors.load_word2vec_format(BIN_PATH, binary=True, mmap='r')

else:
    print('Downloading "word2vec-google-news-300" (this is large; ~1.6GB). It may take several minutes.')
    # gensim.downloader caches downloads but we also save an explicit copy in ../models
    wv = api.load('word2vec-google-news-300')

    # Save a gensim-native KeyedVectors object for faster subsequent loads
    print('Saving KeyedVectors to', KV_PATH)
    wv.save(KV_PATH)

    # Also save the original word2vec binary format for compatibility
    print('Saving word2vec binary to', BIN_PATH)
    wv.save_word2vec_format(BIN_PATH, binary=True)

    print('Saved both formats to', MODEL_DIR)


In [ ]:
# Load the model using memory-mapping where possible to reduce RAM usage.
# If the `wv` variable is already set by the download cell above, we reuse it.
from gensim.models import KeyedVectors

if 'wv' not in globals():
    if os.path.exists(KV_PATH):
        # Gensim-native KeyedVectors — fast to load with .load
        wv = KeyedVectors.load(KV_PATH, mmap='r')
    elif os.path.exists(BIN_PATH):
        # Original binary format — compatible with many tools
        wv = KeyedVectors.load_word2vec_format(BIN_PATH, binary=True, mmap='r')
    else:
        raise FileNotFoundError('No saved model found. Run the download cell first.')

# Quick sanity checks
print('vocabulary size:', len(wv.index_to_key))
print('vector size:', wv.vector_size)


In [ ]:
# Example: find words most similar to 'cricket'.
# `most_similar` returns a list of (word, similarity) tuples.
print(wv.most_similar('cricket', topn=10))


[('cricketing', 0.8372224569320679),
 ('cricketers', 0.8165746331214905),
 ('Test_cricket', 0.8094819784164429),
 ('Twenty##_cricket', 0.8068488836288452),
 ('Twenty##', 0.762426495552063),
 ('Cricket', 0.75413978099823),
 ('cricketer', 0.7372579574584961),
 ('twenty##', 0.7316358089447021),
 ('T##_cricket', 0.7304614186286926),
 ('West_Indies_cricket', 0.6987985968589783)]

In [ ]:
# Compute similarity between two words using cosine similarity.
# Values range from -1 (opposite) to 1 (identical meaning).
print('similarity(cricket, sports)=', wv.similarity('cricket', 'sports'))


np.float32(0.40087256)

In [ ]:
# Analogy example: king - man + woman ≈ queen
# We construct the vector arithmetic and inspect the resulting vector.
vec = wv['king'] - wv['man'] + wv['woman']
# `vec` is a raw numpy array representing the target vector; the next cell
# will query the model for the nearest words to this vector.
vec


array([ 4.29687500e-02, -1.78222656e-01, -1.29089355e-01,  1.15234375e-01,
        2.68554688e-03, -1.02294922e-01,  1.95800781e-01, -1.79504395e-01,
        1.95312500e-02,  4.09919739e-01, -3.68164062e-01, -3.96484375e-01,
       -1.56738281e-01,  1.46484375e-03, -9.30175781e-02, -1.16455078e-01,
       -5.51757812e-02, -1.07574463e-01,  7.91015625e-02,  1.98974609e-01,
        2.38525391e-01,  6.34002686e-02, -2.17285156e-02,  0.00000000e+00,
        4.72412109e-02, -2.17773438e-01, -3.44726562e-01,  6.37207031e-02,
        3.16406250e-01, -1.97631836e-01,  8.59375000e-02, -8.11767578e-02,
       -3.71093750e-02,  3.15551758e-01, -3.41796875e-01, -4.68750000e-02,
        9.76562500e-02,  8.39843750e-02, -9.71679688e-02,  5.17578125e-02,
       -5.00488281e-02, -2.20947266e-01,  2.29492188e-01,  1.26403809e-01,
        2.49023438e-01,  2.09960938e-02, -1.09863281e-01,  5.81054688e-02,
       -3.35693359e-02,  1.29577637e-01,  2.41699219e-02,  3.48129272e-02,
       -2.60009766e-01,  

In [ ]:
# Use the analogy vector to find nearest words in the embedding space.
# Passing a list with the vector tells gensim to treat it as a query vector.
print(wv.most_similar([vec], topn=10))


[('king', 0.8449392914772034),
 ('queen', 0.7300517559051514),
 ('monarch', 0.6454660296440125),
 ('princess', 0.6156251430511475),
 ('crown_prince', 0.5818676948547363),
 ('prince', 0.5777117609977722),
 ('kings', 0.5613663792610168),
 ('sultan', 0.5376776456832886),
 ('Queen_Consort', 0.5344247221946716),
 ('queens', 0.5289887189865112)]

In [ ]:
# End of notebook — model is saved under ../models.
# Reminders:
# - Don't commit the ../models folder to Git; add it to .gitignore.
# - Use mmap='r' when loading to avoid high memory usage.
